In [2]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content="This is the first document.",
        metadata={"source": "doc1.txt"}
    ),
    Document(
        page_content="This is the second document.",
        metadata={"source": "doc2.txt"}
    )
]
documents

[Document(metadata={'source': 'doc1.txt'}, page_content='This is the first document.'),
 Document(metadata={'source': 'doc2.txt'}, page_content='This is the second document.')]

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant")

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [9]:
from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(documents, embedding)

In [10]:
await vectorstore.asimilarity_search("What is the content of the first document?", k=1)

[Document(id='3bd608f3-3e12-4257-8729-5e4c35966c27', metadata={'source': 'doc1.txt'}, page_content='This is the first document.')]

In [11]:
vectorstore.similarity_search_with_score("What is the content of the first document?", k=1)

[(Document(id='3bd608f3-3e12-4257-8729-5e4c35966c27', metadata={'source': 'doc1.txt'}, page_content='This is the first document.'),
  0.40694719552993774)]

Retrievers

LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method.

In [14]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

retriever.batch(["ai","agent"])

[[Document(id='c2fb174c-a764-4ca8-b2a4-02d81c521bfe', metadata={'source': 'doc2.txt'}, page_content='This is the second document.')],
 [Document(id='c2fb174c-a764-4ca8-b2a4-02d81c521bfe', metadata={'source': 'doc2.txt'}, page_content='This is the second document.')]]

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message="""
Answer this question using the provided context only.
{question}
Context:
{context}
"""
prompt=ChatPromptTemplate.from_messages([("human",message)])

rag_chain={'context':retriever,"question":RunnablePassthrough()}|prompt|llm
response=rag_chain.invoke("tell me about first doc")
response


AIMessage(content="The first document has the following details:\n\n- ID: '3bd608f3-3e12-4257-8729-5e4c35966c27'\n- Source: 'doc1.txt'\n- Page content: 'This is the first document.'", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 100, 'total_tokens': 157, 'completion_time': 0.056777286, 'completion_tokens_details': None, 'prompt_time': 0.008044386, 'prompt_tokens_details': None, 'queue_time': 0.047141823, 'total_time': 0.064821672}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d59b8-7ea6-7b63-b300-ecb1f039da09-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 100, 'output_tokens': 57, 'total_tokens': 157})